# Process DMS data

Import Python modules

In [1]:
import os
import glob
import pandas as pd
import numpy as np
import glob
from collections import defaultdict
import math
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(font_scale=1.0, style='ticks', palette='colorblind')
import scipy
from Bio import SeqIO
from Bio.Seq import Seq
import subprocess

## HA DMS from Yu et al.

Align the sequence from the DMS experiment to the reference sequence used to build the H3 tree.

In [ ]:
# Read in the sequence of the reference HA used to build the tree
# TODO: pass path using papermill
ref_fasta = '../../flu-usher/results_260226/HA/H3/curated_reference.fasta'
ref_seq = SeqIO.read(ref_fasta, 'fasta').seq
ref_aa_seq = str(ref_seq.translate())
print(len(ref_aa_seq))

# Read in the HA sequence from the DMS experiment
numbering_df = pd.read_csv('../data/dms_data/Yu_HA/site_numbering_map.csv')
aa_seq = ''.join(list(numbering_df['sequential_wt']))

# Save sequences to a FASTA file for alignment
output_dir = '../results/dms_data/Yu_HA/'
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)
unaligned_fasta = os.path.join(output_dir, 'unaligned.fasta')
if not os.path.isfile(unaligned_fasta):
    with open(unaligned_fasta, 'w') as f:
        f.write('>reference\n')
        f.write(ref_aa_seq + '\n')
        f.write('>dms\n')
        f.write(aa_seq + '\n')

# Align the sequences using MUSCLE
aligned_fasta = f'{output_dir}/aligned.fasta'
if not os.path.isfile(aligned_fasta):
    cmd = ['muscle', '-align', unaligned_fasta, '-output', aligned_fasta]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)

567


Read in the aligned sequences and determine numbering scheme

In [8]:
# Read in the aligned sequences
seqs_dict = defaultdict(list)
aligned_records = list(SeqIO.parse(aligned_fasta, 'fasta'))
for record in aligned_records:
    seqs_dict[record.id] = str(record.seq)

aligned_ref_seq = seqs_dict['reference']
aligned_dms_seq = seqs_dict['dms']

# Compute percent identity
n_sites_to_compare = 0
n_identical = 0
for (alignment_index, (ref_aa, dms_aa)) in enumerate(zip(aligned_ref_seq, aligned_dms_seq), 1):
    if ref_aa != '-' and dms_aa != '-':
        n_sites_to_compare += 1
        if ref_aa == dms_aa:
            n_identical += 1

print(n_sites_to_compare, n_identical, n_identical / n_sites_to_compare)

# Determine numbering scheme
numbering_dict = defaultdict(list)
assert len(aligned_ref_seq) == len(aligned_dms_seq)
alignment_length = len(aligned_ref_seq)
for (seq_id, seq) in [('tree_reference_site', aligned_ref_seq), ('sequential_site', aligned_dms_seq)]:
    seq_index = 1
    for (alignment_index, aa) in enumerate(seq, 1):
        if aa != '-':
            numbering_dict['alignment_index'].append(alignment_index)
            numbering_dict[f'seq_id'].append(seq_id)
            numbering_dict[f'seq_index'].append(seq_index)
            numbering_dict[f'seq_aa'].append(aa)
            seq_index += 1

alignment_numbering_df = (
    pd.DataFrame(numbering_dict)
    .pivot(index='alignment_index', columns='seq_id', values='seq_index')
    .reset_index()
    .rename_axis(None, axis=1)
    .dropna()
    [['sequential_site', 'tree_reference_site', ]]
    .merge(numbering_df)
)
alignment_numbering_df['sequential_site'] = alignment_numbering_df['sequential_site'].astype(int)
alignment_numbering_df['tree_reference_site'] = alignment_numbering_df['tree_reference_site'].astype(int)
del alignment_numbering_df['region']
alignment_numbering_df.head()

504 425 0.8432539682539683


,sequential_site,tree_reference_site,reference_site,sequential_wt,rbs_region
0,1,17,1,Q,outside RBS
1,2,18,2,K,outside RBS
2,3,19,3,I,outside RBS
3,4,20,4,P,outside RBS
4,5,21,5,G,outside RBS


Read in the DMS data and add columns that map from DMS numbering to alignment numbering.

In [17]:
# Read in DMS data and merge with numbering data and fitness data
f = '../data/dms_data/Yu_HA/Phenotypes.csv'
ha_dms_data = (
    pd.read_csv(f)
    .rename(
        columns={
            'MDCKSIAT1 cell entry' : 'dms_effect',
            'sera escape' : 'sera_escape',
            'pH stability' : 'pH_stability',
            'wildtype' : 'wt_aa',
            'mutant' : 'mut_aa',
            'nt changes to codon' : 'n_nt_changes'
        }
    )
    .merge(alignment_numbering_df, on='sequential_site', validate='many_to_one')
    # drop rows where dms_effect is NA
    .dropna(subset=['dms_effect'])
)
del ha_dms_data['region']
del ha_dms_data['rbs_region']
ha_dms_data.to_csv('../results/dms_data/Yu_HA/processed_dms_data.csv', index=False)
print("Number of mutations with processed data", len(ha_dms_data))
ha_dms_data.head()

Number of mutations with processed data 9738


,site,wt_aa,mut_aa,sera_escape,dms_effect,pH_stability,sequential_site,n_nt_changes,tree_reference_site,reference_site,sequential_wt
0,1,Q,A,0.08825,-0.1226,0.004237,1,2,17,1,Q
1,1,Q,C,0.01779,-0.5732,-0.014300,1,3,17,1,Q
2,1,Q,D,-0.05395,0.2550,-0.021900,1,2,17,1,Q
3,1,Q,E,-0.01963,0.2941,0.006890,1,1,17,1,Q
4,1,Q,F,-0.16350,-0.7141,-0.001402,1,3,17,1,Q


## NP data from Bloom et al.

The NP sequence used in the DMS experiment and the one used as the reference for tree building should be the same (Aichi 1968). Read in the DMS data and make sure that this is the case. Then, compute mutational effects.

In [19]:
# Read in the sequence of the reference NP used to build the tree
# TODO: update path using papermill
ref_fasta = '../../flu-usher/results_260226/NP/all/curated_reference.fasta'
ref_seq = SeqIO.read(ref_fasta, 'fasta').seq
ref_aa_seq = str(ref_seq.translate())
ref_aa_seq = ref_aa_seq.replace('*', '')
print('Sequence length:', len(ref_aa_seq))

# Read in the DMS data
np_dms_data = pd.read_excel('../data/dms_data/Bloom_NP/Supplementary_file_1.xls')
col_dict = {
    '#SITE' : 'site', 'WT_AA' : 'wt_aa'}
aa_cols = []
for col in np_dms_data.columns:
    if 'PI' in col:
        aa = col[-1]
        col_dict[col] = aa
        aa_cols.append(aa)
np_dms_data.rename(columns=col_dict, inplace=True)

# Correct apparent error in data
np_dms_data['wt_aa'] = np_dms_data['wt_aa'].apply(lambda x: x.replace('N,N,H,H', 'N'))

# Get the sequence of the unmutated protein and make sure that it matches the reference
aa_seq = ''.join(list(np_dms_data['wt_aa']))
assert aa_seq == ref_aa_seq
print('Sequence from DMS data matches the reference sequence used to build the tree')

# Melt dataframe
np_dms_data = pd.melt(
    np_dms_data, id_vars=['site', 'wt_aa'], value_vars=aa_cols,
    var_name='mut_aa', value_name='preference'
)

# Make a smaller dataframe that only has data for rows where the wt_aa == mut_aa
wt_data = np_dms_data[np_dms_data['wt_aa'] == np_dms_data['mut_aa']].copy()
wt_data.rename(columns={'preference' : 'wt_preference'}, inplace=True)

# Merge the wt preferences into the dataframe with all preferences
np_dms_data = np_dms_data.merge(wt_data[['site', 'wt_preference']], on='site')

# Add a column that quantifies the mutation's effect (=preference/wt_preference)
np_dms_data['dms_effect'] = np.log(np_dms_data['preference'] / np_dms_data['wt_preference'])

# Save the data to an output file
output_dir = '../results/dms_data/Bloom_NP/'
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)
output_file = os.path.join(output_dir, 'processed_dms_data.csv')
if not os.path.isfile(output_file):
    np_dms_data.to_csv(output_file, index=False)

np_dms_data.head()

Sequence length: 498
Sequence from DMS data matches the reference sequence used to build the tree


,site,wt_aa,mut_aa,preference,wt_preference,dms_effect
0,1,M,A,0.026720,0.391903,-2.685591
1,2,A,A,0.753436,0.753436,0.000000
2,3,S,A,0.059008,0.116818,-0.682944
3,4,Q,A,0.020215,0.029481,-0.377335
4,5,G,A,0.114320,0.213952,-0.626750


## PB2 data from Soh et el.

Align the unmutated DMS sequence and the sequence used as reference for tree building. See if there are any gaps.

In [18]:
# Read in the sequence of the reference NP used to build the tree
# TODO: update path using papermill
ref_fasta = '../../flu-usher/results_260226/PB2/all/curated_reference.fasta'
ref_seq = SeqIO.read(ref_fasta, 'fasta').seq
ref_aa_seq = str(ref_seq.translate())
ref_aa_seq = ref_aa_seq.replace('*', '')
print('reference sequence length:', len(ref_aa_seq))

# Read in the DMS data and get the sequence of the unmutated protein
pb2_dms_data = (
    pd.read_csv('../data/dms_data/Soh_PB2/elife-45079-fig2-data1-v1.csv')
    .rename(columns={
        'log2effectA549' : 'dms_effect', 
        'wildtype' : 'wt_aa', 
        'mutation' : 'mut_aa'
    })
)
aa_seq = ''.join(list(
    pb2_dms_data
    .sort_values('site')
    .drop_duplicates('site')
    ['wt_aa']
))
print('DMS sequence length:', len(aa_seq))

# Save sequences to a FASTA file for alignment
output_dir = '../results/dms_data/Soh_PB2/'
if not os.path.isdir(output_dir):
    os.makedirs(output_dir)
unaligned_fasta = os.path.join(output_dir, 'unaligned.fasta')
if not os.path.isfile(unaligned_fasta):
    with open(unaligned_fasta, 'w') as f:
        f.write('>reference\n')
        f.write(ref_aa_seq + '\n')
        f.write('>dms\n')
        f.write(aa_seq + '\n')

# Align the sequences using MUSCLE
aligned_fasta = f'{output_dir}/aligned.fasta'
if not os.path.isfile(aligned_fasta):
    cmd = ['muscle', '-align', unaligned_fasta, '-output', aligned_fasta]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)

# Read in the aligned sequences
seqs_dict = defaultdict(list)
aligned_records = list(SeqIO.parse(aligned_fasta, 'fasta'))
for record in aligned_records:
    seqs_dict[record.id] = str(record.seq)

aligned_ref_seq = seqs_dict['reference']
aligned_dms_seq = seqs_dict['dms']

if '-' not in aligned_dms_seq and '-' not in aligned_ref_seq:
    print('No gaps in the alignment')

# Compute the percent identity
id = sum([a == b for (a, b) in zip(aligned_ref_seq, aligned_dms_seq)]) / len(aligned_ref_seq)
print('Percent identity:', id)

reference sequence length: 759
DMS sequence length: 759
No gaps in the alignment
Percent identity: 0.9591567852437418


## NA data from Wang et al.

In [20]:
# Read in the sequence of the reference NA used to build the N1 tree
ref_fasta = '../../flu-usher/results_260226/NA/N1/curated_reference.fasta'
ref_seq = SeqIO.read(ref_fasta, 'fasta').seq
ref_aa_seq = str(ref_seq.translate())
ref_aa_seq = ref_aa_seq.replace('*', '')
print('reference sequence length:', len(ref_aa_seq))

# Read in the DMS data and get the sequence of the unmutated protein
na_dms_data = pd.read_excel('../data/dms_data/Wang_NA/msystems.00670-23-s0006.xlsx')
na_dms_data['site'] = na_dms_data['AA change'].apply(lambda x: int(x[1:-1]))
na_dms_data['wt_aa'] = na_dms_data['AA change'].apply(lambda x: x[0])
na_dms_data['mut_aa'] = na_dms_data['AA change'].apply(lambda x: x[-1])

aa_seq = ''.join(list(
    na_dms_data
    .sort_values('site')
    .drop_duplicates('site')
    ['wt_aa']
))
print('DMS sequence length:', len(aa_seq))

reference sequence length: 454
DMS sequence length: 401
